In [2]:
import pandas as pd, numpy as np, joblib, os, re, json
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

dataset = "legal_cases_dataset_fixed.csv"

df = pd.read_csv(dataset)

print("\nsample row (first):")
display(df.head(1).T)



sample row (first):


,0
case_number,S.L.P.(Crl.) No. 58155/2011
domain,Cyber Crime
case_type,identity theft
scenario_text,Anahi Rastogi filed a complaint involving iden...
solution_summary,The recommended legal approach involves invoki...
tags,"cybercrime,it_act_66c,data_protection,identity..."


In [3]:
def clean_text(s):
    if not isinstance(s, str):
        return ""
    s = s.replace('\n', ' ').strip()
    s = re.sub(r'\s+', ' ', s)
    return s.lower()

df['scenario_clean'] = df['scenario_text'].apply(clean_text)
df['scenario_len_chars'] = df['scenario_clean'].str.len()
df['scenario_len_words'] = df['scenario_clean'].str.split().apply(len)


In [4]:
tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    stop_words='english'
)

X = tfidf.fit_transform(df['scenario_clean'].fillna(''))

joblib.dump(tfidf, "tfidf_vectorizer.joblib")

from scipy import sparse
sparse.save_npz("tfidf_matrix.npz", X)


In [5]:
nn = NearestNeighbors(n_neighbors=10, metric='cosine', algorithm='auto', n_jobs=-1)
nn.fit(X)
joblib.dump(nn, "nn_model.joblib")


['nn_model.joblib']

In [6]:
from typing import List, Tuple

def get_similar_cases(input_text: str, k: int = 5) -> List[Tuple[str, float, str, str]]:
    cleaned = clean_text(input_text)
    v = tfidf.transform([cleaned])
    distances, indices = nn.kneighbors(v, n_neighbors=k)
    distances = distances.flatten()
    indices = indices.flatten()
    
    similarities = 1 - distances
    results = []
    
    for idx, sim in zip(indices, similarities):
        row = df.iloc[idx]
        results.append((
            row["case_number"],
            float(sim),
            row.get("tags", ""),
            row.get("solution_summary", "")
        ))

    return results


In [7]:
example = df["scenario_text"].iloc[0]
res = get_similar_cases(example, k=3)
res


[('S.L.P.(Crl.) No. 58155/2011',
  1.0,
  'cybercrime,it_act_66c,data_protection,identity_theft',
  'The recommended legal approach involves invoking the applicable provisions of Indian law related to identity theft. The affected party may seek relief through appropriate procedures under CPC. Further remedies may include compensation, injunction, or additional statutory protections.'),
 ('Crl.A. 5373/2013',
  0.4615132039134656,
  'cybercrime,it_act_66c,data_protection,identity_theft',
  'The recommended legal approach involves invoking the applicable provisions of Indian law related to identity theft. The affected party may seek relief through appropriate procedures under CPC. Further remedies may include compensation, injunction, or additional statutory protections.'),
 ('Crl.A. 6085/2020',
  0.4549348437588796,
  'cybercrime,it_act_66c,data_protection,identity_theft',
  'The recommended legal approach involves invoking the applicable provisions of Indian law related to identity thef

In [8]:
use_default = input("Use sample scenario? (y/n): ").lower().strip()

if use_default == "y":
    user_input = "A tenant claims illegal eviction by landlord without notice and seeks remedy under rent control provisions."
    print("\nUsing sample input:\n", user_input)
else:
    user_input = input("\nEnter your legal scenario:\n").strip()

results = get_similar_cases(user_input, k=5)

print("\n---- Top similar cases ----")
for rank, (case_number, sim, tags, sol) in enumerate(results, start=1):
    print(f"\nRank {rank}: case_number={case_number}, similarity={sim:.4f}")
    row = df[df['case_number'] == case_number].iloc[0]
    print("Scenario (truncated):", row['scenario_text'][:400])
    print("Tags:", tags)
    print("Solution:", sol)



Using sample input:
 A tenant claims illegal eviction by landlord without notice and seeks remedy under rent control provisions.

---- Top similar cases ----

Rank 1: case_number=Civil Appeal No. 8330 of 2018, similarity=0.1960
Scenario (truncated): Eva Keer filed a complaint involving illegal possession under Property Dispute. The incident occurred in Panihati, where the accused allegedly violated relevant provisions such as Consumer Protection Act Sec 2. The matter is currently at the stage where a petition has been submitted to the court. Parties are disputing facts, and the authorities have initiated preliminary hearings to determine liab
Tags: property_dispute,rera,transfer_of_property_act,illegal_possession
Solution: The recommended legal approach involves invoking the applicable provisions of Indian law related to illegal possession. The affected party may seek relief through appropriate procedures under CPC. Further remedies may include compensation, injunction, or additional 

In [9]:
df = pd.read_csv("lawyers_dataset_10000.csv")
df.head()

,lawyer_id,lawyer_name,practice_area,case_types,experience_years,city,contact_email,contact_phone
0,L00001,Kavita Mehta,Intellectual Property,Trademark|Copyright|Patent,30,Chennai,kavita.mehta1@lawsynthetic.com,9321859392
1,L00002,Aarti Tandon,Intellectual Property,Trademark|Copyright|Patent,17,Kolkata,aarti.tandon2@lawsynthetic.com,9529953323
2,L00003,Vikram Sharma,Labor Law,Wrongful Termination|Salary Dispute|Labor Comp...,24,Chennai,vikram.sharma3@lawsynthetic.com,9815348826
3,L00004,Nitin Mishra,Intellectual Property,Trademark|Copyright|Patent,28,Noida,nitin.mishra4@lawsynthetic.com,9328268445
4,L00005,Arjun Verma,Labor Law,Wrongful Termination|Salary Dispute|Labor Comp...,18,Hyderabad,arjun.verma5@lawsynthetic.com,9459369038


In [10]:
df.shape


(10000, 8)

In [11]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   lawyer_id         10000 non-null  object
 1   lawyer_name       10000 non-null  object
 2   practice_area     10000 non-null  object
 3   case_types        10000 non-null  object
 4   experience_years  10000 non-null  int64 
 5   city              10000 non-null  object
 6   contact_email     10000 non-null  object
 7   contact_phone     10000 non-null  int64 
dtypes: int64(2), object(6)
memory usage: 625.1+ KB


In [12]:
df.describe()


,experience_years,contact_phone
count,10000.000000,1.000000e+04
mean,15.720600,9.546855e+09
std,8.642789,2.616848e+08
min,1.000000,9.100149e+09
25%,8.000000,9.318234e+09
50%,16.000000,9.545641e+09
75%,23.000000,9.772839e+09
max,30.000000,9.999929e+09


In [13]:
df["practice_area"].value_counts()


practice_area
Corporate Law            1056
Intellectual Property    1047
Immigration Law          1047
Cyber Law                1014
Labor Law                1005
Criminal Law             1000
Civil Law                 976
Tax Law                   972
Family Law                944
Consumer Law              939
Name: count, dtype: int64

In [14]:
df["practice_area"].value_counts()


practice_area
Corporate Law            1056
Intellectual Property    1047
Immigration Law          1047
Cyber Law                1014
Labor Law                1005
Criminal Law             1000
Civil Law                 976
Tax Law                   972
Family Law                944
Consumer Law              939
Name: count, dtype: int64

In [15]:
df["city"].value_counts()


city
Mumbai       1044
Jaipur       1041
Gurgaon      1033
Hyderabad    1019
Bangalore    1006
Chennai       989
Noida         971
Pune          970
Delhi         968
Kolkata       959
Name: count, dtype: int64

In [16]:
df["text"] = df["practice_area"] + " " + df["case_types"]


In [19]:
tfidf = TfidfVectorizer(stop_words="english")

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X = df["text"]
y = df["practice_area"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

y_pred = model.predict(X_test_vec)
accuracy_score(y_test, y_pred)



1.0

In [20]:
from sklearn.metrics.pairwise import cosine_similarity

lawyer_vectors = tfidf.transform(df["text"])

def recommend_lawyers(user_case, top_n=5):
    user_vec = tfidf.transform([user_case])
    similarity_scores = cosine_similarity(user_vec, lawyer_vectors).flatten()
    
    top_indices = similarity_scores.argsort()[-top_n:][::-1]
    
    return df.loc[top_indices, [
        "lawyer_name", "practice_area",
        "experience_years", "city", "contact_email"
    ]]


In [21]:
recommend_lawyers("divorce and child custody case")


,lawyer_name,practice_area,experience_years,city,contact_email
5811,Arjun Mehta,Family Law,26,Gurgaon,arjun.mehta5812@lawsynthetic.com
7233,Aarti Mehta,Family Law,13,Pune,aarti.mehta7234@lawsynthetic.com
8011,Aarti Malhotra,Family Law,27,Chennai,aarti.malhotra8012@lawsynthetic.com
8678,Kavita Verma,Family Law,24,Bangalore,kavita.verma8679@lawsynthetic.com
6654,Priya Mehta,Family Law,5,Chennai,priya.mehta6655@lawsynthetic.com


In [30]:
import pandas as pd
import re

df = pd.read_csv("lawyers_dataset_10000.csv") 
def clean_text(s):
    if not isinstance(s, str):
        return ""
    s = s.replace("\n", " ").strip()
    s = re.sub(r"\s+", " ", s)
    return s.lower()

df["scenario_clean"] = (
    df["practice_area"].fillna("") + " " +
    df["case_types"].fillna("")
).apply(clean_text)

df.to_csv("lawyers_dataset_nn_ready.csv", index=False)






In [31]:
tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    stop_words="english"
)

X = tfidf.fit_transform(df["scenario_clean"].fillna(""))


In [32]:
joblib.dump(tfidf, "lawyer_tfidf_vectorizer.joblib")
sparse.save_npz("lawyer_tfidf_matrix.npz", X)


In [33]:
nn = NearestNeighbors(
    n_neighbors=10,
    metric="cosine",
    algorithm="auto",
    n_jobs=-1
)

nn.fit(X)


,n_neighbors,10
,radius,1.0
,algorithm,'auto'
,leaf_size,30
,metric,'cosine'
,p,2
,metric_params,None
,n_jobs,-1


In [34]:
joblib.dump(nn, "lawyer_nn_model.joblib")


['lawyer_nn_model.joblib']